In [2]:
spark

In [17]:
from pyspark.sql.functions import col

In [3]:
sc.addPyFile('/opt/spark-3.5.1/jars/graphframes-0.8.4-spark3.5-s_2.12.jar')

In [4]:
from graphframes import GraphFrame

In [6]:
#Motifidying <-- Search for a Pattern in a Dataframe  <-- Subgraph <-- Filteration


In [7]:
#Vertix Dataframe
v = spark.createDataFrame([
    ('a', 'Alice', 34,'DS',3000),
    ('b', 'Bob', 36,'ML',2000),
    ('c', 'Charlie', 30,'WebDev',800),
    ('d', 'David', 29,'Programmer',1000),
    ('e', 'Esther', 32,'None',0),
    ('f', 'Fanny', 36,'ML',20000),
    ('g', 'Gabby', 60,'Retired',5000)
], ['id', 'name', 'age' , 'job','salary'])
#Edge Dataframe
e = spark.createDataFrame([
    ('a', 'b', 'friend',2),
    ('b', 'c', 'follow',6),
    ('c', 'b', 'follow',7),
    ('f', 'c', 'follow',0),
    ('e', 'f', 'follow',1),
    ('e', 'd', 'friend',2),
    ('d', 'a', 'friend',5),
    ('a', 'e', 'friend',9)
], ['src', 'dst', 'relationship', 'strength'])
g = GraphFrame(v,e)

In [15]:
#Find Patterns in Graph

m1 = g.find('(v1)-[e1]->(v2)')

In [16]:
m1.printSchema()

root
 |-- v1: struct (nullable = false)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: long (nullable = true)
 |    |-- job: string (nullable = true)
 |    |-- salary: long (nullable = true)
 |-- e1: struct (nullable = false)
 |    |-- src: string (nullable = true)
 |    |-- dst: string (nullable = true)
 |    |-- relationship: string (nullable = true)
 |    |-- strength: long (nullable = true)
 |-- v2: struct (nullable = false)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: long (nullable = true)
 |    |-- job: string (nullable = true)
 |    |-- salary: long (nullable = true)



In [25]:
d2 = m1.filter(col('v1')['age']>30)

In [26]:
d2.show(truncate=False)

+-------------------------+-----------------+--------------------------------+
|v1                       |e1               |v2                              |
+-------------------------+-----------------+--------------------------------+
|{f, Fanny, 36, ML, 20000}|{f, c, follow, 0}|{c, Charlie, 30, WebDev, 800}   |
|{b, Bob, 36, ML, 2000}   |{b, c, follow, 6}|{c, Charlie, 30, WebDev, 800}   |
|{a, Alice, 34, DS, 3000} |{a, b, friend, 2}|{b, Bob, 36, ML, 2000}          |
|{a, Alice, 34, DS, 3000} |{a, e, friend, 9}|{e, Esther, 32, None, 0}        |
|{e, Esther, 32, None, 0} |{e, d, friend, 2}|{d, David, 29, Programmer, 1000}|
|{e, Esther, 32, None, 0} |{e, f, follow, 1}|{f, Fanny, 36, ML, 20000}       |
+-------------------------+-----------------+--------------------------------+



In [27]:
d3 = m1.filter(col('v1').age > col('v2').age)

In [29]:
d3.show(truncate = False)

+-------------------------+-----------------+--------------------------------+
|v1                       |e1               |v2                              |
+-------------------------+-----------------+--------------------------------+
|{f, Fanny, 36, ML, 20000}|{f, c, follow, 0}|{c, Charlie, 30, WebDev, 800}   |
|{b, Bob, 36, ML, 2000}   |{b, c, follow, 6}|{c, Charlie, 30, WebDev, 800}   |
|{a, Alice, 34, DS, 3000} |{a, e, friend, 9}|{e, Esther, 32, None, 0}        |
|{e, Esther, 32, None, 0} |{e, d, friend, 2}|{d, David, 29, Programmer, 1000}|
+-------------------------+-----------------+--------------------------------+



In [31]:
df4 = m1.filter(col('e1').relationship == 'friend')

In [34]:
df4.show(truncate = False)

+--------------------------------+-----------------+--------------------------------+
|v1                              |e1               |v2                              |
+--------------------------------+-----------------+--------------------------------+
|{d, David, 29, Programmer, 1000}|{d, a, friend, 5}|{a, Alice, 34, DS, 3000}        |
|{a, Alice, 34, DS, 3000}        |{a, b, friend, 2}|{b, Bob, 36, ML, 2000}          |
|{a, Alice, 34, DS, 3000}        |{a, e, friend, 9}|{e, Esther, 32, None, 0}        |
|{e, Esther, 32, None, 0}        |{e, d, friend, 2}|{d, David, 29, Programmer, 1000}|
+--------------------------------+-----------------+--------------------------------+



In [36]:
%time
m1.cache()

CPU times: user 2 μs, sys: 1 μs, total: 3 μs
Wall time: 5.25 μs


DataFrame[v1: struct<id:string,name:string,age:bigint,job:string,salary:bigint>, e1: struct<src:string,dst:string,relationship:string,strength:bigint>, v2: struct<id:string,name:string,age:bigint,job:string,salary:bigint>]

In [46]:
#BiDirectional Pattern

m2 = g.find('(v1)-[e1]->(v2);(v2)-[e2]->(v1)')

In [48]:
m2.show(truncate = False)

+-----------------------------+-----------------+-----------------------------+-----------------+
|v1                           |e1               |v2                           |e2               |
+-----------------------------+-----------------+-----------------------------+-----------------+
|{c, Charlie, 30, WebDev, 800}|{c, b, follow, 7}|{b, Bob, 36, ML, 2000}       |{b, c, follow, 6}|
|{b, Bob, 36, ML, 2000}       |{b, c, follow, 6}|{c, Charlie, 30, WebDev, 800}|{c, b, follow, 7}|
+-----------------------------+-----------------+-----------------------------+-----------------+



In [50]:
#TriDyud Pattern

m3 =  g.find ('(v1)-[e1]->(v2);(v2)-[e2]->(v3);(v3)-[e3]->(v1)')

In [54]:
#Dispaly notebook cell with horizontal Scroll bar
from IPython.display import display , HTML
display(HTML("<style>pre {white-space: pre !important; }</style>"))

In [52]:
m3.show(truncate=False)

+--------------------------------+-----------------+--------------------------------+-----------------+--------------------------------+-----------------+
|v1                              |e1               |v2                              |e2               |v3                              |e3               |
+--------------------------------+-----------------+--------------------------------+-----------------+--------------------------------+-----------------+
|{d, David, 29, Programmer, 1000}|{d, a, friend, 5}|{a, Alice, 34, DS, 3000}        |{a, e, friend, 9}|{e, Esther, 32, None, 0}        |{e, d, friend, 2}|
|{a, Alice, 34, DS, 3000}        |{a, e, friend, 9}|{e, Esther, 32, None, 0}        |{e, d, friend, 2}|{d, David, 29, Programmer, 1000}|{d, a, friend, 5}|
|{e, Esther, 32, None, 0}        |{e, d, friend, 2}|{d, David, 29, Programmer, 1000}|{d, a, friend, 5}|{a, Alice, 34, DS, 3000}        |{a, e, friend, 9}|
+--------------------------------+-----------------+------------------

In [58]:
#drop unNecessary Columns
m4 =  g.find ('(v1)-[]->(v2);(v2)-[]->(v3);(v3)-[]->(v1)')
m4.show(truncate=False)

+--------------------------------+--------------------------------+--------------------------------+
|v1                              |v2                              |v3                              |
+--------------------------------+--------------------------------+--------------------------------+
|{d, David, 29, Programmer, 1000}|{a, Alice, 34, DS, 3000}        |{e, Esther, 32, None, 0}        |
|{a, Alice, 34, DS, 3000}        |{e, Esther, 32, None, 0}        |{d, David, 29, Programmer, 1000}|
|{e, Esther, 32, None, 0}        |{d, David, 29, Programmer, 1000}|{a, Alice, 34, DS, 3000}        |
+--------------------------------+--------------------------------+--------------------------------+



In [64]:
# Pattern tree nodes
m5=  g.find ('(v1)-[]->(v2);(v2)-[]->(v3)')
m5.show(truncate=False)# Pattern tree nodes


+--------------------------------+--------------------------------+--------------------------------+
|v1                              |v2                              |v3                              |
+--------------------------------+--------------------------------+--------------------------------+
|{e, Esther, 32, None, 0}        |{d, David, 29, Programmer, 1000}|{a, Alice, 34, DS, 3000}        |
|{d, David, 29, Programmer, 1000}|{a, Alice, 34, DS, 3000}        |{b, Bob, 36, ML, 2000}          |
|{f, Fanny, 36, ML, 20000}       |{c, Charlie, 30, WebDev, 800}   |{b, Bob, 36, ML, 2000}          |
|{b, Bob, 36, ML, 2000}          |{c, Charlie, 30, WebDev, 800}   |{b, Bob, 36, ML, 2000}          |
|{c, Charlie, 30, WebDev, 800}   |{b, Bob, 36, ML, 2000}          |{c, Charlie, 30, WebDev, 800}   |
|{a, Alice, 34, DS, 3000}        |{b, Bob, 36, ML, 2000}          |{c, Charlie, 30, WebDev, 800}   |
|{e, Esther, 32, None, 0}        |{f, Fanny, 36, ML, 20000}       |{c, Charlie, 30, WebDev,

In [65]:
#You can Drop Not Common Vertices
m5=  g.find ('()-[]->(v2);(v2)-[]->()')
m5.show(truncate=False)

+--------------------------------+
|v2                              |
+--------------------------------+
|{b, Bob, 36, ML, 2000}          |
|{b, Bob, 36, ML, 2000}          |
|{a, Alice, 34, DS, 3000}        |
|{f, Fanny, 36, ML, 20000}       |
|{c, Charlie, 30, WebDev, 800}   |
|{c, Charlie, 30, WebDev, 800}   |
|{e, Esther, 32, None, 0}        |
|{e, Esther, 32, None, 0}        |
|{d, David, 29, Programmer, 1000}|
|{a, Alice, 34, DS, 3000}        |
+--------------------------------+



In [66]:
m6=  g.find ('(v1)-[]->(v2);(v2)-[]->(v3)')
m6.show(truncate=False)

+--------------------------------+--------------------------------+--------------------------------+
|v1                              |v2                              |v3                              |
+--------------------------------+--------------------------------+--------------------------------+
|{e, Esther, 32, None, 0}        |{d, David, 29, Programmer, 1000}|{a, Alice, 34, DS, 3000}        |
|{d, David, 29, Programmer, 1000}|{a, Alice, 34, DS, 3000}        |{b, Bob, 36, ML, 2000}          |
|{f, Fanny, 36, ML, 20000}       |{c, Charlie, 30, WebDev, 800}   |{b, Bob, 36, ML, 2000}          |
|{b, Bob, 36, ML, 2000}          |{c, Charlie, 30, WebDev, 800}   |{b, Bob, 36, ML, 2000}          |
|{c, Charlie, 30, WebDev, 800}   |{b, Bob, 36, ML, 2000}          |{c, Charlie, 30, WebDev, 800}   |
|{a, Alice, 34, DS, 3000}        |{b, Bob, 36, ML, 2000}          |{c, Charlie, 30, WebDev, 800}   |
|{e, Esther, 32, None, 0}        |{f, Fanny, 36, ML, 20000}       |{c, Charlie, 30, WebDev,

In [68]:
m6=  g.find ('(v1)-[]->(v2);(v2)-[]->(v3)').filter(col('v3').id != col('v1').id)
m6.show(truncate=False)

+--------------------------------+--------------------------------+--------------------------------+
|v1                              |v2                              |v3                              |
+--------------------------------+--------------------------------+--------------------------------+
|{e, Esther, 32, None, 0}        |{d, David, 29, Programmer, 1000}|{a, Alice, 34, DS, 3000}        |
|{d, David, 29, Programmer, 1000}|{a, Alice, 34, DS, 3000}        |{b, Bob, 36, ML, 2000}          |
|{f, Fanny, 36, ML, 20000}       |{c, Charlie, 30, WebDev, 800}   |{b, Bob, 36, ML, 2000}          |
|{a, Alice, 34, DS, 3000}        |{b, Bob, 36, ML, 2000}          |{c, Charlie, 30, WebDev, 800}   |
|{e, Esther, 32, None, 0}        |{f, Fanny, 36, ML, 20000}       |{c, Charlie, 30, WebDev, 800}   |
|{a, Alice, 34, DS, 3000}        |{e, Esther, 32, None, 0}        |{d, David, 29, Programmer, 1000}|
|{d, David, 29, Programmer, 1000}|{a, Alice, 34, DS, 3000}        |{e, Esther, 32, None, 0}

In [77]:
# Only Return Single Direction not Bi Direction (Negation)

m7 = g.find('(v1)-[e1]->(v2);!(v2)-[]->(v1)')
m7.show(truncate=False)

+--------------------------------+-----------------+--------------------------------+
|v1                              |e1               |v2                              |
+--------------------------------+-----------------+--------------------------------+
|{d, David, 29, Programmer, 1000}|{d, a, friend, 5}|{a, Alice, 34, DS, 3000}        |
|{e, Esther, 32, None, 0}        |{e, d, friend, 2}|{d, David, 29, Programmer, 1000}|
|{a, Alice, 34, DS, 3000}        |{a, b, friend, 2}|{b, Bob, 36, ML, 2000}          |
|{e, Esther, 32, None, 0}        |{e, f, follow, 1}|{f, Fanny, 36, ML, 20000}       |
|{f, Fanny, 36, ML, 20000}       |{f, c, follow, 0}|{c, Charlie, 30, WebDev, 800}   |
|{a, Alice, 34, DS, 3000}        |{a, e, friend, 9}|{e, Esther, 32, None, 0}        |
+--------------------------------+-----------------+--------------------------------+

